In [2]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns


m_df = pd.read_csv(r'C:\Users\acces\OneDrive\Desktop\ML PROJECT\master_late_delivery.csv')

In [3]:
m_df.columns.tolist()

['order_id',
 'customer_id',
 'order_purchase_timestamp',
 'order_estimated_delivery_date',
 'is_late',
 'customer_city',
 'customer_state',
 'total_items',
 'total_price',
 'total_freight_value',
 'avg_price',
 'product_rows',
 'total_weight',
 'total_height',
 'total_length',
 'total_width',
 'max_heigth',
 'max_weight',
 'max_length',
 'max_width',
 'avg_weight',
 'avg_heigth',
 'avg_length',
 'avg_width',
 'total_payments',
 'total_value',
 'max_installments',
 'total_sellers',
 'seller_state_count',
 'same_state_seller_count',
 'seller_state',
 'multiple_seller_state']

In [4]:
m_df.dtypes

order_id                             str
customer_id                          str
order_purchase_timestamp             str
order_estimated_delivery_date        str
is_late                            int64
customer_city                        str
customer_state                       str
total_items                        int64
total_price                      float64
total_freight_value              float64
avg_price                        float64
product_rows                       int64
total_weight                     float64
total_height                     float64
total_length                     float64
total_width                      float64
max_heigth                       float64
max_weight                       float64
max_length                       float64
max_width                        float64
avg_weight                       float64
avg_heigth                       float64
avg_length                       float64
avg_width                        float64
total_payments  

#### CONVERTING COLUMNS FROM STR TO DATETIME FOR THE CALCULATION OF THE DELIVERY DURATION AND EXTRACTING THE DATE AND TIME INTO DIFFERENT COLUMNS FOR FUTURE MODELING

In [5]:
m_df['order_purchase_timestamp'] = pd.to_datetime(m_df['order_purchase_timestamp'])

In [6]:
m_df['purchase_year'] = m_df['order_purchase_timestamp'].dt.year
m_df['purchase_month'] = m_df['order_purchase_timestamp'].dt.month
m_df['purchase_day'] = m_df['order_purchase_timestamp'].dt.day
m_df['purchase_weekday'] = m_df['order_purchase_timestamp'].dt.weekday
m_df['purchase_hour'] = m_df['order_purchase_timestamp'].dt.hour

C:\Users\acces\AppData\Local\Temp\ipykernel_29252\2806465004.py:4: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  m_df['purchase_weekday'] = m_df['order_purchase_timestamp'].dt.weekday


In [7]:
m_df['order_estimated_delivery_date'] = pd.to_datetime(m_df['order_estimated_delivery_date'])

In [8]:
m_df['estimated_delivery_days'] = (m_df['order_estimated_delivery_date'] - m_df['order_purchase_timestamp']).dt.total_seconds()/86400

In [9]:
m_df['total_volume'] = m_df['total_length'] * m_df['total_height'] * m_df['total_width']

In [10]:
m_df['freight_per_kg'] = m_df['total_freight_value'] / m_df['total_weight']/1000

In [11]:
m_df['freight_price_ratio']= m_df['total_freight_value'] / m_df['total_price']

In [12]:
m_df['order_size'] = pd.cut(m_df['product_rows'],bins=[0,1,3,float('inf')], labels=['small','medium','large'])

In [13]:
m_df['weight_category']=pd.cut(m_df['total_weight'],bins=[0,1500,5000,float('inf')],labels=['light','medium','heavy'])

In [14]:
m_df['same_state'] =(m_df['customer_state'] == m_df['seller_state']).astype(int)

#### FEATURES VALIDATION

In [15]:
new_features = ['purchase_year','purchase_month','purchase_day','purchase_weekday','purchase_hour','estimated_delivery_days','total_volume','freight_per_kg','freight_price_ratio','order_size','weight_category','same_state']

In [16]:
m_df[new_features].head()

,purchase_year,purchase_month,purchase_day,purchase_weekday,purchase_hour,estimated_delivery_days,total_volume,freight_per_kg,freight_price_ratio,order_size,weight_category,same_state
0,2018,8,8,2,10,11.582928,2400.0,0.000064,0.984604,small,light,1
1,2017,5,15,0,21,21.095440,2880.0,0.000028,0.579452,small,light,0
2,2017,11,21,1,18,15.212234,816.0,0.000039,0.538881,small,light,1
3,2018,6,11,0,21,8.113194,2873.0,0.000053,0.284588,small,light,1
4,2017,8,15,1,2,21.885208,4576.0,0.000022,0.503501,small,light,0


In [17]:
m_df[new_features].describe(include='all')

c:\Users\acces\AppData\Local\Programs\Python\Python315\Lib\site-packages\pandas\core\nanops.py:1324: RuntimeWarning: invalid value encountered in subtract
  sqr = (avg - values) ** 2


,purchase_year,purchase_month,purchase_day,purchase_weekday,purchase_hour,estimated_delivery_days,total_volume,freight_per_kg,freight_price_ratio,order_size,weight_category,same_state
count,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,96470.000000,9.645400e+04,9.645400e+04,96470.000000,96470,96448,96470.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,3,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,small,light,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86835,65385,NaN
mean,2017.544314,6.031046,15.516420,2.756494,14.773028,23.736343,5.645204e+04,inf,0.308386,NaN,NaN,0.358402
std,0.503562,3.228479,8.665335,1.967041,5.328347,8.761052,9.556565e+05,NaN,0.311618,NaN,NaN,0.479533
min,2016.000000,1.000000,1.000000,0.000000,0.000000,2.008009,1.680000e+02,0.000000e+00,0.000000,NaN,NaN,0.000000
25%,2017.000000,3.000000,8.000000,1.000000,11.000000,18.329905,3.000000e+03,9.695652e-06,0.132017,NaN,NaN,0.000000
50%,2018.000000,6.000000,16.000000,3.000000,15.000000,23.230880,8.000000e+03,2.257143e-05,0.224374,NaN,NaN,0.000000
75%,2018.000000,8.000000,23.000000,4.000000,19.000000,28.407795,2.362500e+04,5.056433e-05,0.380567,NaN,NaN,1.000000


In [18]:
inf_cols = np.isinf(m_df[new_features].select_dtypes(include='number').sum())

#### HANDLING MISSING VALUES AND INFINITY

In [19]:
print(inf_cols)

purchase_year              False
purchase_month             False
purchase_day               False
purchase_weekday           False
purchase_hour              False
estimated_delivery_days    False
total_volume               False
freight_per_kg              True
freight_price_ratio        False
same_state                 False
dtype: bool


In [20]:
m_df[new_features].isna().sum()


purchase_year               0
purchase_month              0
purchase_day                0
purchase_weekday            0
purchase_hour               0
estimated_delivery_days     0
total_volume               16
freight_per_kg             16
freight_price_ratio         0
order_size                  0
weight_category            22
same_state                  0
dtype: int64

In [21]:
m_df['total_weight'].isna().sum()

np.int64(16)

In [22]:
(m_df['total_weight'] == 0).sum()

np.int64(6)

In [23]:
np.isinf(m_df['freight_per_kg']).value_counts()

freight_per_kg
False    96464
True         6
Name: count, dtype: int64

In [24]:
m_df['freight_per_kg'] = m_df['freight_per_kg'].replace(np.inf, np.nan) 

In [25]:
np.isinf(m_df['freight_per_kg']).value_counts()

freight_per_kg
False    96470
Name: count, dtype: int64

In [26]:
m_df['freight_per_kg'].isna().sum()

np.int64(22)

In [27]:
m_df[new_features].isna().sum()

purchase_year               0
purchase_month              0
purchase_day                0
purchase_weekday            0
purchase_hour               0
estimated_delivery_days     0
total_volume               16
freight_per_kg             22
freight_price_ratio         0
order_size                  0
weight_category            22
same_state                  0
dtype: int64

In [28]:
m_df[m_df['total_volume'].isna()][['total_length','total_height','total_width']].head(25).reset_index()

,index,total_length,total_height,total_width
0,738,NaN,NaN,NaN
1,9744,NaN,NaN,NaN
2,33548,NaN,NaN,NaN
3,35468,NaN,NaN,NaN
4,43511,NaN,NaN,NaN
5,43751,NaN,NaN,NaN
6,48918,NaN,NaN,NaN
7,59848,NaN,NaN,NaN
8,69429,NaN,NaN,NaN
9,69848,NaN,NaN,NaN


In [29]:
m_df[m_df['freight_per_kg'].isna()][['total_freight_value','total_weight']].head(30).reset_index()

,index,total_freight_value,total_weight
0,125,47.70,0.0
1,738,16.05,NaN
2,7108,47.42,0.0
3,9744,14.52,NaN
4,12769,19.89,0.0
5,23602,23.71,0.0
6,28949,14.49,0.0
7,33548,29.04,NaN
8,35468,9.27,NaN
9,43511,16.79,NaN


In [30]:
m_df['total_weight'].isna().sum()

np.int64(16)

In [31]:
(m_df['total_weight'] == 0).sum()

np.int64(6)

In [32]:
m_df['missing_weights'] = m_df['total_weight'].isna().astype(int)

In [33]:
m_df['missing_weights'].value_counts()

missing_weights
0    96454
1       16
Name: count, dtype: int64

In [34]:
m_df['weight_zero']=(m_df['total_weight']== 0).astype(int)

In [35]:
m_df['weight_zero'].value_counts()

weight_zero
0    96464
1        6
Name: count, dtype: int64

#### FEATURE SELECTION

In [36]:
features_column = ['purchase_month','purchase_day','purchase_weekday','purchase_hour','estimated_delivery_days','total_weight','total_volume','product_rows','max_weight','total_price','total_freight_value','freight_per_kg','freight_price_ratio','customer_state','seller_state','same_state','order_size','weight_category','missing_weights','weight_zero']

In [37]:
x = m_df[features_column]
y = m_df['is_late']

In [38]:
x.shape

(96470, 20)

In [39]:
y.shape

(96470,)

In [40]:
x.dtypes

purchase_month                int32
purchase_day                  int32
purchase_weekday              int32
purchase_hour                 int32
estimated_delivery_days     float64
total_weight                float64
total_volume                float64
product_rows                  int64
max_weight                  float64
total_price                 float64
total_freight_value         float64
freight_per_kg              float64
freight_price_ratio         float64
customer_state                  str
seller_state                    str
same_state                    int64
order_size                 category
weight_category            category
missing_weights               int64
weight_zero                   int64
dtype: object

In [41]:
model_df = m_df[features_column + ['is_late']]
model_df.to_csv('model_data.csv',index=False)